In [1]:
import os

os.environ['HADOOP_USER_NAME'] = 'hdfs'

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Jupyter_HDFS_Native_Read") \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://master:9000") \
    .getOrCreate()

hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
print("fs.defaultFS:", hadoop_conf.get("fs.defaultFS"))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/11 09:35:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


fs.defaultFS: hdfs://master:9000


In [3]:
import sys
sys.path.append("src")

In [4]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType, DoubleType, TimestampType
from pathlib import Path
import json

TYPE_MAP = {
    "integer": IntegerType(),
    "string": StringType(),
    "float": FloatType(),
    "double": DoubleType(),
    "timestamp": TimestampType()
}

def load_schema_spark(table: str) -> StructType:
    schema_path = Path("config/schemas") / f"{table}_schema.json"
    with open(schema_path) as f:
        schema_def = json.load(f)
    fields = []
    for col in schema_def["columns"]:
        spark_type = TYPE_MAP.get(col["type"], StringType())
        fields.append(StructField(col["name"], spark_type, col["nullable"]))
    return StructType(fields)

In [116]:
table_path = "hdfs://master:9000/data/bronze/distribution_centers/distribution_centers.csv"

schema = load_schema_spark("distribution_centers")

df_cen = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv(table_path)

df_cen.show(20, truncate=False)

+---+-------------------------------------------+--------+---------+
|id |name                                       |latitude|longitude|
+---+-------------------------------------------+--------+---------+
|1  |Memphis TN                                 |35.1174 |-89.9711 |
|2  |Chicago IL                                 |41.8369 |-87.6847 |
|3  |Houston TX                                 |29.7604 |-95.3698 |
|4  |Los Angeles CA                             |34.05   |-118.25  |
|5  |New Orleans LA                             |29.95   |-90.0667 |
|6  |Port Authority of New York/New Jersey NY/NJ|40.634  |-73.7834 |
|7  |Philadelphia PA                            |39.95   |-75.1667 |
|8  |Mobile AL                                  |30.6944 |-88.0431 |
|9  |Charleston SC                              |32.7833 |-79.9333 |
|10 |Savannah GA                                |32.0167 |-81.1167 |
+---+-------------------------------------------+--------+---------+



In [8]:
df.count()

10

In [10]:
len(df.columns)

4

In [14]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)



In [19]:
from pyspark.sql.functions import col
df.filter(col("id").isNull()).count()

0

In [27]:
df.groupby("id")\
    .count()\
    .filter(col("count") > 1)\
    .show()

+---+-----+
| id|count|
+---+-----+
+---+-----+



In [32]:
df.filter(col("id").isNotNull()).show()

+---+--------------------+--------+---------+
| id|                name|latitude|longitude|
+---+--------------------+--------+---------+
|  1|          Memphis TN| 35.1174| -89.9711|
|  2|          Chicago IL| 41.8369| -87.6847|
|  3|          Houston TX| 29.7604| -95.3698|
|  4|      Los Angeles CA|   34.05|  -118.25|
|  5|      New Orleans LA|   29.95| -90.0667|
|  6|Port Authority of...|  40.634| -73.7834|
|  7|     Philadelphia PA|   39.95| -75.1667|
|  8|           Mobile AL| 30.6944| -88.0431|
|  9|       Charleston SC| 32.7833| -79.9333|
| 10|         Savannah GA| 32.0167| -81.1167|
+---+--------------------+--------+---------+



In [37]:
df.select("name").distinct().show(truncate = False)

+-------------------------------------------+
|name                                       |
+-------------------------------------------+
|Chicago IL                                 |
|Charleston SC                              |
|New Orleans LA                             |
|Savannah GA                                |
|Port Authority of New York/New Jersey NY/NJ|
|Los Angeles CA                             |
|Philadelphia PA                            |
|Memphis TN                                 |
|Houston TX                                 |
|Mobile AL                                  |
+-------------------------------------------+



In [41]:
df.groupby("name").count().filter(col("count")>1) .show()

+----+-----+
|name|count|
+----+-----+
+----+-----+



In [45]:
from pyspark.sql.functions import col, trim

space_issues = df.filter(col("name") != trim(col("name")))

space_issues.show(truncate=False)


+---+----+--------+---------+
|id |name|latitude|longitude|
+---+----+--------+---------+
+---+----+--------+---------+



In [117]:
table_path = "hdfs://master:9000/data/bronze/products/products.csv"

schema = load_schema_spark("products")

df_pro = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv(table_path)

# بديل لـ df.show()
df_pro.limit(5).toPandas()

,id,cost,category,name,brand,retail_price,department,sku,distribution_center_id
0,13842,2.51875,Accessories,Low Profile Dyed Cotton Twill Cap - Navy W39S55D,MG,6.25,Women,EBD58B8A3F1D72F4206201DA62FB1204,1
1,13928,2.33835,Accessories,Low Profile Dyed Cotton Twill Cap - Putty W39S55D,MG,5.95,Women,2EAC42424D12436BDD6A5B8A88480CC3,1
2,14115,4.87956,Accessories,Enzyme Regular Solid Army Caps-Black W35S45D,MG,10.99,Women,EE364229B2791D1EF9355708EFF0BA34,1
3,14157,4.64877,Accessories,Enzyme Regular Solid Army Caps-Olive W35S45D (...,MG,10.99,Women,00BD13095D06C20B11A2993CA419D16B,1
4,14273,6.50793,Accessories,Washed Canvas Ivy Cap - Black W11S64C,MG,15.99,Women,F531DC20FDE20B7ADF3A73F52B71D0AF,1


In [50]:
df.count()

29120

In [52]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- cost: float (nullable = true)
 |-- category: string (nullable = true)
 |-- name: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- retail_price: float (nullable = true)
 |-- department: string (nullable = true)
 |-- sku: string (nullable = true)
 |-- distribution_center_id: integer (nullable = true)



In [53]:
df.filter(col("id").isNull()).count()

0

In [56]:
df.groupby("id").count().filter(col("count")>1).show()


+---+-----+
| id|count|
+---+-----+
+---+-----+



In [72]:
df_cost = df.filter((col("cost") <= 0) | (col("cost").isNull()))
df_cost.show()

+---+----+--------+----+-----+------------+----------+---+----------------------+
| id|cost|category|name|brand|retail_price|department|sku|distribution_center_id|
+---+----+--------+----+-----+------------+----------+---+----------------------+
+---+----+--------+----+-----+------------+----------+---+----------------------+



In [77]:
df.select("category").distinct().show(50, truncate=False)

+-----------------------------+
|category                     |
+-----------------------------+
|Dresses                      |
|Swim                         |
|Outerwear & Coats            |
|Tops & Tees                  |
|Suits & Sport Coats          |
|Socks & Hosiery              |
|Blazers & Jackets            |
|Suits                        |
|Active                       |
|Pants                        |
|Pants & Capris               |
|Clothing Sets                |
|Leggings                     |
|Skirts                       |
|Intimates                    |
|Fashion Hoodies & Sweatshirts|
|Plus                         |
|Accessories                  |
|Maternity                    |
|Underwear                    |
|Sleep & Lounge               |
|Shorts                       |
|Sweaters                     |
|Jumpsuits & Rompers          |
|Socks                        |
|Jeans                        |
+-----------------------------+



In [79]:
from pyspark.sql import functions as F

# الصح: دالة trim بتاكل العمود جواها الفانكشن نفسها
df_spaces = df.filter(F.col("category") != F.trim(F.col("category")))

df_spaces.show(truncate=False)


+---+----+--------+----+-----+------------+----------+---+----------------------+
|id |cost|category|name|brand|retail_price|department|sku|distribution_center_id|
+---+----+--------+----+-----+------------+----------+---+----------------------+
+---+----+--------+----+-----+------------+----------+---+----------------------+



In [89]:
df.select("name").distinct().count()

27310

In [90]:
df.count()

29120

In [92]:
df.filter(F.col("name") != F.trim(F.col("name"))).show()

+---+----+--------+----+-----+------------+----------+---+----------------------+
| id|cost|category|name|brand|retail_price|department|sku|distribution_center_id|
+---+----+--------+----+-----+------------+----------+---+----------------------+
+---+----+--------+----+-----+------------+----------+---+----------------------+



In [95]:
df.groupby("name").count().filter(F.col("count")>1).show(truncate = False)

+------------------------------------------------------------------------------+-----+
|name                                                                          |count|
+------------------------------------------------------------------------------+-----+
|State O Maine Big and Tall Solid Microfleece Lounge Pant                      |2    |
|Alfred Dunner Women's Proportioned Medium Pant                                |3    |
|Bikini Babydoll Set-Black                                                     |2    |
|Kenneth Cole Men's Straight Leg Jean                                          |3    |
|Genuine Leather Simple Check Book Holder style - mw156CF Brown                |2    |
|Black Pleated Stretch Waist Lambskin Leather Skirt - Tyra                     |2    |
|Wallflower Junior PLUS SIZE Skinny Clean Jegging Jean                         |2    |
|Smartwool Women's Midweight Pattern Crew                                      |2    |
|Hanes Classics Men's 6-Pack Cushion Low Cu

In [101]:
df.filter(F.col("name") == "State O Maine Big and Tall Solid Microfleece Lounge Pant" ).limit(5).toPandas()

,id,cost,category,name,brand,retail_price,department,sku,distribution_center_id
0,26857,9.68941,Sleep & Lounge,State O Maine Big and Tall Solid Microfleece L...,KNOTHE CORP.,26.99,Men,EA9BE6EA49C5C752ABB11953955C90E4,1
1,26909,11.74065,Sleep & Lounge,State O Maine Big and Tall Solid Microfleece L...,KNOTHE CORP.,26.99,Men,5B948B56FF119949410B6BD8ACAE9B66,1


In [102]:
df.groupby("name").count().filter(F.col("count")>1).count()

1568

In [103]:
df.filter(F.col("name") == "Alfred Dunner Women's Proportioned Medium Pant" ).limit(5).toPandas()

,id,cost,category,name,brand,retail_price,department,sku,distribution_center_id
0,5272,13.500,Pants & Capris,Alfred Dunner Women's Proportioned Medium Pant,Alfred Dunner,25.0,Women,E8855B3528CB03D1DEF9803220BD3CB9,2
1,5418,21.868,Pants & Capris,Alfred Dunner Women's Proportioned Medium Pant,Alfred Dunner,44.0,Women,7D3E28D14440D6C07F73B7557E3D9602,2
2,5522,24.156,Pants & Capris,Alfred Dunner Women's Proportioned Medium Pant,Alfred Dunner,44.0,Women,132D6C1408F2492456848667346B54B6,2


In [104]:
df.groupby("brand").count().filter(F.col("count") > 1).show()

+-----------------+-----+
|            brand|count|
+-----------------+-----+
|            C1RCA|    3|
|       Horny Toad|   11|
|      Evan Picone|   36|
|      Jules & Jim|   10|
|    Sock It To Me|   56|
|Cecilia de Rafael|    4|
|         Stitch's|    3|
|         Rawlings|    2|
|       Silly yogi|    4|
|      Union Jeans|    7|
|          Varsity|    7|
|        Wolverine|   27|
| Sweetheart Slips|   17|
|           Vobaga|   12|
| Oxfords Cashmere|    3|
|            Nobis|    4|
|      NurtureWear|    2|
|     Ivory Falcon|    2|
|             Domo|    3|
|         Coolibar|   27|
+-----------------+-----+
only showing top 20 rows



In [109]:
df.select("brand").distinct().count()

2757

In [110]:
df.filter(F.col("brand") != F.trim(F.col("brand"))).show()

+---+----+--------+----+-----+------------+----------+---+----------------------+
| id|cost|category|name|brand|retail_price|department|sku|distribution_center_id|
+---+----+--------+----+-----+------------+----------+---+----------------------+
+---+----+--------+----+-----+------------+----------+---+----------------------+



In [114]:
df.filter(
    (F.col("retail_price") < F.col("cost")) |
    (F.col("retail_price").isNull()) |
    (F.col("retail_price") < 0)
).show()

+---+----+--------+----+-----+------------+----------+---+----------------------+
| id|cost|category|name|brand|retail_price|department|sku|distribution_center_id|
+---+----+--------+----+-----+------------+----------+---+----------------------+
+---+----+--------+----+-----+------------+----------+---+----------------------+



In [115]:
df.select("department").distinct().show()

+----------+
|department|
+----------+
|       Men|
|     Women|
+----------+



In [128]:
df_pro.groupby("distribution_center_id").count().orderBy("distribution_center_id", ascending=True).show()

+----------------------+-----+
|distribution_center_id|count|
+----------------------+-----+
|                     1| 3891|
|                     2| 3929|
|                     3| 3667|
|                     4| 2761|
|                     5| 2112|
|                     6| 2572|
|                     7| 2669|
|                     8| 2919|
|                     9| 2719|
|                    10| 1881|
+----------------------+-----+



In [130]:
df.filter(F.col("distribution_center_id").isNull()).show()

+---+----+--------+----+-----+------------+----------+---+----------------------+
| id|cost|category|name|brand|retail_price|department|sku|distribution_center_id|
+---+----+--------+----+-----+------------+----------+---+----------------------+
+---+----+--------+----+-----+------------+----------+---+----------------------+

